# Overpass Waypoints

This notebook is designed to manage and store waypoint data extracted via the Overpass API. The waypoints represent specific latitude and longitude coordinates associated with hiking routes. The extracted data is processed and ingested into a SQL database for further analysis and integration with other datasets. Key operations include creating the `OVRP_waypoints` table, transforming waypoint data, and uploading it to a cloud-based SQL database.

In [1]:
import overpy
import pandas as pd 
import requests
import json
import pymssql
from sqlalchemy import Integer, String, Float, DATETIME, create_engine

In [2]:
# Load configuration from config/db_config.json
with open('../config/db_config.json', 'r') as f:
    db_config = json.load(f)

# Get database credentials
server = db_config['server']
database = db_config['database']
db_user = db_config['db_user']
db_password = db_config['db_password']# Connect to SQL Database
conn = pymssql.connect(server, db_user, db_password, database)

# Create connection string for SQLAlchemy
connection_string = f"mssql+pymssql://{db_user}:{db_password}@{server}/{database}"
engine = create_engine(connection_string)

In [ ]:
# SQL query
query = "SELECT DISTINCT id FROM OVRP_HikingRoutes"

# load data from database
df_hikingroute_ids = pd.read_sql_query(query, con=engine)

# convert to list
list_hikingroute_ids = df_hikingroute_ids['id'].tolist()

# show list
print(f"Die Liste enthält {len(list_hikingroute_ids)} Elemente.")  # Anzahl ausgeben

Die Liste enthält 3139 Elemente.


In [6]:
# connect to Overpass API
api = overpy.Overpass(url="http://overpass.osm.ch/api/interpreter")

# timestamp for API calls
timestamp_apicall = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

# empty list for results
results = []

# query Overpass API
for route_id in list_hikingroute_ids:
    
    query = f"""
    [out:json];
    relation["route"="hiking"](id:{route_id});
    way(r);
    out ids center tags;
    """
    try:
        # send query
        result = api.query(query)

        # iterate over all ways
        for way in result.ways:
            if way.center_lat and way.center_lon:
                # save results
                results.append({
                    "id": way.id,
                    "itshikingroute": route_id,
                    "lat": way.center_lat,
                    "lon": way.center_lon,
                    "timestamp_apicall": timestamp_apicall
                })

    except overpy.exception.OverpassException as e:
        print(f"Fehler bei der Anfrage für ID {route_id}: {e}")

# transform to dataframe
df_waypoints = pd.DataFrame(results)
df_waypoints['lat'] = pd.to_numeric(df_waypoints['lat'], errors='coerce')
df_waypoints['lon'] = pd.to_numeric(df_waypoints['lon'], errors='coerce')
df_waypoints['timestamp_apicall'] = pd.to_datetime(df_waypoints['timestamp_apicall'], errors='coerce')

# show results
print(df_waypoints.head(3))

         id  itshikingroute        lat       lon   timestamp_apicall
0  28022787          120125  46.871293  6.581740 2024-12-10 19:48:17
1  28022788          120125  46.874111  6.589236 2024-12-10 19:48:17
2  28022789          120125  46.878365  6.598630 2024-12-10 19:48:17


In [ ]:
# Create table if it doesn't exist
query = f"""
        CREATE TABLE OVRP_waypoints(
            id                      INT         NOT NULL,
            itshikingroute          INT         NOT NULL,
            lat                     FLOAT       NOT NULL,
            lon                     FLOAT       NOT NULL,
            timestamp_apicall       DATETIME    NULL,
            FOREIGN KEY (itshikingroute) REFERENCES OVRP_HikingRoutes (id)
        );
    """

conn = pymssql.connect(server, db_user, db_password, database)
cursor = conn.cursor()
cursor.execute(query)

conn.commit()
conn.close()

In [ ]:
table_name = "OVRP_waypoints"

# Create connection string for SQLAlchemy
connection_string = f"mssql+pymssql://{db_user}:{db_password}@{server}/{database}"
engine = create_engine(connection_string)

# Ingest data to tabledatabase table
df_waypoints.to_sql(table_name, con=engine, if_exists='append', index=False)
print("DataFrame erfolgreich in die MSSQL-Datenbank geladen!")

DataFrame erfolgreich in die MSSQL-Datenbank geladen!
